In [1]:
import os
import json
import time
import requests
from dotenv import load_dotenv

In [2]:
load_dotenv()
headers = {"Authorization": f"Bearer {os.getenv('TMDB_TOKEN')}"}

URL_FILM = "https://api.themoviedb.org/3/movie/{}"   # {} sera remplacé par l'id du film
POSTES = ["Director", "Screenplay", "Writer", "Producer", "Original Music Composer"]

# Relecture des identifiants sauvegardés par le notebook précédent
with open("../data/ids_films.json") as fichier:
    ids_films = json.load(fichier)

print(len(ids_films), "identifiants chargés")

6015 identifiants chargés



**fonction** réutilisable :
elle télécharge un film, filtre le casting, et renvoie le résultat.

In [3]:
def telecharger_film(id_film):
    """Télécharge un film avec son casting filtré. Renvoie None en cas d'erreur."""
    url = URL_FILM.format(id_film)
    params = {"language": "fr-FR", "append_to_response": "credits"}
    reponse = requests.get(url, headers=headers, params=params)

    if reponse.status_code != 200:
        print(f"Erreur {reponse.status_code} pour le film {id_film}")
        return None

    film = reponse.json()

    # Filtrage du casting : 10 premiers acteurs + postes clés
    film["credits"]["cast"] = film["credits"]["cast"][:10]
    film["credits"]["crew"] = [m for m in film["credits"]["crew"] if m["job"] in POSTES]

    return film

In [4]:
for id_film in ids_films[:3]:
    film = telecharger_film(id_film)
    print(film["title"], "|",
          len(film["credits"]["cast"]), "acteurs |",
          len(film["credits"]["crew"]), "membres d'équipe")

Interstellar | 10 acteurs | 7 membres d'équipe
Inception | 10 acteurs | 7 membres d'équipe
Avengers | 10 acteurs | 4 membres d'équipe


## 3. Téléchargement complet avec reprise
Chaque film est écrit dans `data/films_details.jsonl` dès qu'il est téléchargé
(format **JSON Lines** : une ligne = un film au format JSON).

Si le téléchargement s'interrompt, on relance simplement la cellule :
les films déjà présents dans le fichier sont ignorés.

In [5]:
FICHIER = "../data/films_details.jsonl"

# 1. Quels films ont déjà été téléchargés ?
deja_faits = set()
if os.path.exists(FICHIER):
    with open(FICHIER, encoding="utf-8") as f:
        for ligne in f:
            deja_faits.add(json.loads(ligne)["id"])

a_faire = [i for i in ids_films if i not in deja_faits]
print("Déjà téléchargés :", len(deja_faits), "| Restants :", len(a_faire))

Déjà téléchargés : 0 | Restants : 6015


In [6]:
erreurs = []
debut = time.time()

# "a" = append : on ajoute à la fin du fichier sans effacer ce qui existe
with open(FICHIER, "a", encoding="utf-8") as f:
    for n, id_film in enumerate(a_faire, start=1):
        try:
            film = telecharger_film(id_film)
        except requests.RequestException as e:   # coupure réseau, timeout…
            print("Erreur réseau pour", id_film, ":", e)
            film = None

        if film is None:
            erreurs.append(id_film)
        else:
            f.write(json.dumps(film, ensure_ascii=False) + "\n")   # une ligne par film

        if n % 200 == 0:   # point d'étape tous les 200 films
            minutes = (time.time() - debut) / 60
            print(f"{n}/{len(a_faire)} films — {minutes:.1f} min")

        time.sleep(0.05)   # petite pause pour ménager l'API

print("Terminé ! Erreurs :", len(erreurs))

200/6015 films — 0.7 min
400/6015 films — 1.3 min
600/6015 films — 2.0 min
800/6015 films — 2.7 min
1000/6015 films — 3.3 min
1200/6015 films — 3.9 min
1400/6015 films — 4.6 min
1600/6015 films — 5.2 min
1800/6015 films — 5.9 min
2000/6015 films — 6.5 min
2200/6015 films — 7.2 min
2400/6015 films — 7.8 min
2600/6015 films — 8.5 min
2800/6015 films — 9.1 min
3000/6015 films — 9.8 min
3200/6015 films — 10.4 min
3400/6015 films — 11.1 min
3600/6015 films — 11.7 min
3800/6015 films — 12.4 min
4000/6015 films — 13.1 min
4200/6015 films — 13.7 min
4400/6015 films — 14.3 min
4600/6015 films — 15.0 min
4800/6015 films — 15.6 min
5000/6015 films — 16.3 min
5200/6015 films — 16.9 min
5400/6015 films — 17.6 min
5600/6015 films — 18.2 min
5800/6015 films — 18.8 min
6000/6015 films — 19.5 min
Terminé ! Erreurs : 0


In [7]:
# Nombre de films dans le fichier
with open(FICHIER, encoding="utf-8") as f:
    films = [json.loads(ligne) for ligne in f]

print("Films dans le fichier :", len(films))
print("Identifiants uniques :", len({film["id"] for film in films}))
print("Taille du fichier :", round(os.path.getsize(FICHIER) / 1_000_000, 1), "Mo")

Films dans le fichier : 6015
Identifiants uniques : 6015
Taille du fichier : 42.2 Mo


In [8]:
# Premier aperçu de la qualité des données : combien de budgets / recettes manquants ?
sans_budget = sum(1 for film in films if film["budget"] == 0)
sans_recettes = sum(1 for film in films if film["revenue"] == 0)

print("Films sans budget renseigné :", sans_budget)
print("Films sans recettes renseignées :", sans_recettes)

Films sans budget renseigné : 1526
Films sans recettes renseignées : 1204


### Observations
- 6 015 films téléchargés, aucun doublon, 42,2 Mo.
- 1 526 films (≈ 25 %) n'ont pas de budget renseigné et 1 204 (≈ 20 %) pas de recettes :
  dans TMDB, la valeur 0 signifie « inconnu ».
- Ces valeurs devront être traitées à l'étape 2 (remplacement par une valeur nulle).